In [ ]:
# Prepare model
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertForSequenceClassification.from_pretrained("bert-base-chinese", num_labels=2)
model.eval()

In [ ]:
# optimistic/pessimistic
test_texts = [
    "这部电影太精彩了，剧情和演技都超棒！",
    "这个产品质量很差，用了两天就坏了。"
]

inputs = tokenizer(test_texts, padding=True, truncation=True, return_tensors="pt")
print("分词后的输入：", inputs)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = torch.softmax(logits, dim=1)
pred_labels = torch.argmax(probs, dim=1).numpy()

In [ ]:
label_map = {0: "pessimistic", 1: "optimistic"}
for text, prob, label_id in zip(test_texts, probs, pred_labels):
    print(f"text{text}")
    print(f"predictred label:{label_map[label_id]}, probability:{probs[label_id].item():.4f}\n")

In [ ]:
# Save model and tokenizer
save_dir = "./bert_chinese_classifier"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"model_path: {save_dir}")

# Load model and tokenizer
loaded_model = BertForSequenceClassification.from_pretrained(save_dir)
loaded_tokenizer = BertTokenizer.from_pretrained(save_dir)
loaded_model.eval()
print("Finish loading")

In [ ]:
# Optimize model
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear, torch.nn.LayerNorm},
    dtype=torch.qint8
)

In [ ]:
# Compare speed after optimization
import time

def test_inference_speed(model, inputs, runs=100):
    model.eval()
    with torch.no_grad():
        for _ in range(10):
            model(**inputs)
        start = time.time()

        for _ in range(runs):
            model(**inputs)
        end = time.time()
    return (end - start) / runs

original_speed = test_inference_speed(model, inputs)
quantized_speed = test_inference_speed(quantized_model, inputs)

print(f"time of original model{original_speed*1000:.4f}ms")
print(f"time of quantized model{quantized_speed*1000:.4f}ms")
print(f"{original_speed/quantized_speed:.4f}times quicker than original model")

quantized_model.save_pretrained("./bert_quantized")